# Algoritmos básicos · Miscelánea numérica

Problemas cortos de teoría de números y de manipulación de secuencias: sucesiones $k$-nacci, sumas de dos cuadrados, los números de Hilbert (donde la factorización en primos deja de ser única), la sucesión de Thue-Morse, los récords de huecos entre primos, la función $\varphi$ de Euler, la fórmula de Legendre para $n!$, los primos cubanos, el sistema factorádico y algunas transformaciones de listas. Cada resultado se comprueba con fuerza bruta, una segunda forma de calcularlo o valores conocidos. Ideas del capítulo «Miscelánea» de [*Ejercicios de programación con Python*](https://github.com/jaalonso/Ejercicios-Python), de José A. Alonso Jiménez.

In [1]:
import math
import random
from collections import Counter, deque
from functools import cache
from itertools import accumulate, chain, count, islice, pairwise, permutations, product, repeat, zip_longest


def criba(n):
    es = bytearray([1]) * (n + 1)
    es[0] = es[1] = 0
    for p in range(2, math.isqrt(n) + 1):
        if es[p]:
            es[p * p :: p] = bytes(len(range(p * p, n + 1, p)))
    return es


def es_primo(n):
    return n >= 2 and all(n % d for d in range(2, math.isqrt(n) + 1))


def factoriza(n):
    """Diccionario {primo: exponente} por división por tentativa."""
    factores, d = Counter(), 2
    while d * d <= n:
        while n % d == 0:
            factores[d] += 1
            n //= d
        d += 1
    if n > 1:
        factores[n] += 1
    return factores

## 1. Sucesiones $k$-nacci

En la sucesión de Fibonacci cada término es la suma de los 2 anteriores; en la de Pentanacci, de los 5 anteriores, empezando por $0, 0, 0, 0, 1$. Escribir un generador `k_nacci(k)` que empiece por $k - 1$ ceros y un 1 y produzca cada término nuevo con una ventana deslizante (`deque` de longitud $k$) cuya suma se actualiza restando el término que sale y sumando el que entra, sin volver a sumar los $k$ valores. Comprobarlo para $k = 2, \dots, 7$ contra la definición recursiva memoizada y mostrar los 20 primeros números de Pentanacci y cómo el cociente entre términos consecutivos se acerca a 2 al crecer $k$.

In [2]:
def k_nacci(k):
    ventana = deque([0] * (k - 1) + [1], maxlen=k)
    suma = 1
    yield from ventana
    while True:
        nuevo = suma
        suma += nuevo - ventana[0]
        ventana.append(nuevo)
        yield nuevo


def k_nacci_recursivo(k, n):
    @cache
    def t(i):
        if i < k - 1:
            return 0
        if i == k - 1:
            return 1
        return sum(t(i - j) for j in range(1, k + 1))

    return [t(i) for i in range(n)]


print("Pentanacci:", list(islice(k_nacci(5), 20)))
for k in range(2, 8):
    assert list(islice(k_nacci(k), 300)) == k_nacci_recursivo(k, 300)
assert list(islice(k_nacci(2), 10)) == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

print(f"\n{'k':>2} {'t(400)/t(399)':>15}")
for k in range(2, 11):
    a, b = islice(k_nacci(k), 399, 401)
    print(f"{k:>2} {b / a:>15.10f}")

Pentanacci: [0, 0, 0, 0, 1, 1, 2, 4, 8, 16, 31, 61, 120, 236, 464, 912, 1793, 3525, 6930, 13624]

 k   t(400)/t(399)
 2    1.6180339887
 3    1.8392867552
 4    1.9275619755
 5    1.9659482366
 6    1.9835828434
 7    1.9919641966
 8    1.9960311797
 9    1.9980294703
10    1.9990186327


## 2. Teorema de Navidad de Fermat

Fermat enunció en una carta del 25 de diciembre de 1640 que un primo impar $p$ es suma de dos cuadrados si y solo si $p \equiv 1 \pmod 4$. Escribir `es_suma_de_dos_cuadrados(n)`, que pruebe cada $x$ con $x^2 \le n/2$ y mire si $n - x^2$ es un cuadrado perfecto con `math.isqrt`, y comprobar el teorema para todos los primos impares menores que $10^5$. Mostrar la descomposición de los primeros primos de cada clase.

In [3]:
def es_cuadrado(n):
    return math.isqrt(n) ** 2 == n


def es_suma_de_dos_cuadrados(n):
    return any(es_cuadrado(n - x * x) for x in range(math.isqrt(n // 2) + 1))


es = criba(100_000)
primos_impares = [p for p in range(3, 100_001) if es[p]]
for p in primos_impares:
    assert es_suma_de_dos_cuadrados(p) == (p % 4 == 1)

conteo = Counter(p % 4 for p in primos_impares)
print(f"Primos impares < 10⁵: {conteo[1]} son 4k+1 (todos suma de dos cuadrados), {conteo[3]} son 4k+3 (ninguno)")
for p in primos_impares[:10]:
    x = next((x for x in range(math.isqrt(p // 2) + 1) if es_cuadrado(p - x * x)), None)
    texto = f"{x}² + {math.isqrt(p - x * x)}²" if x is not None else "no es suma de dos cuadrados"
    print(f"{p:>3} ≡ {p % 4} (mod 4): {texto}")

Primos impares < 10⁵: 4783 son 4k+1 (todos suma de dos cuadrados), 4808 son 4k+3 (ninguno)
  3 ≡ 3 (mod 4): no es suma de dos cuadrados
  5 ≡ 1 (mod 4): 1² + 2²
  7 ≡ 3 (mod 4): no es suma de dos cuadrados
 11 ≡ 3 (mod 4): no es suma de dos cuadrados
 13 ≡ 1 (mod 4): 2² + 3²
 17 ≡ 1 (mod 4): 1² + 4²
 19 ≡ 3 (mod 4): no es suma de dos cuadrados
 23 ≡ 3 (mod 4): no es suma de dos cuadrados
 29 ≡ 1 (mod 4): 2² + 5²
 31 ≡ 3 (mod 4): no es suma de dos cuadrados


## 3. Todas las representaciones como suma de dos cuadrados

Escribir `representaciones(n)`, que devuelva las parejas $(x, y)$ con $0 \le x \le y$ y $x^2 + y^2 = n$ en $O(\sqrt n)$ usando `math.isqrt`. Comprobarla con fuerza bruta (todas las parejas) hasta 2000, y además con la fórmula de Jacobi: el número de soluciones con signos y orden, $r_2(n)$, vale $4\,(d_1(n) - d_3(n))$, donde $d_1$ y $d_3$ cuentan los divisores de $n$ de la forma $4k+1$ y $4k+3$. Encontrar el menor número con 1, 2, 3, 4 y 5 representaciones distintas.

In [4]:
def representaciones(n):
    return [(x, math.isqrt(n - x * x)) for x in range(math.isqrt(n // 2) + 1) if es_cuadrado(n - x * x)]


def representaciones_bruta(n):
    return [(x, y) for x in range(n + 1) for y in range(x, n + 1) if x * x + y * y == n]


def r2_jacobi(n):
    divisores = [d for d in range(1, n + 1) if n % d == 0]
    return 4 * (sum(d % 4 == 1 for d in divisores) - sum(d % 4 == 3 for d in divisores))


def r2_desde_representaciones(n):
    """Cuenta los (±x, ±y) y (±y, ±x) que salen de cada pareja x ≤ y."""
    total = 0
    for x, y in representaciones(n):
        signos = (1 if x == 0 else 2) * (1 if y == 0 else 2)
        total += signos * (1 if x == y else 2)
    return total


for n in range(1, 2001):
    if n <= 300:
        assert representaciones(n) == representaciones_bruta(n)
    assert r2_desde_representaciones(n) == r2_jacobi(n)

for n in (2026, 2025, 5525):
    print(f"{n} = " + " = ".join(f"{x}² + {y}²" for x, y in representaciones(n)))

menor = {}
for n in count(1):
    menor.setdefault(len(representaciones(n)), n)
    if all(k in menor for k in range(1, 6)):
        break
print({k: menor[k] for k in range(1, 6)})

2026 = 1² + 45²
2025 = 0² + 45² = 27² + 36²
5525 = 7² + 74² = 14² + 73² = 22² + 71² = 25² + 70² = 41² + 62² = 50² + 55²
{1: 1, 2: 25, 3: 325, 4: 1105, 5: 4225}


## 4. Primos de Hilbert

Los números de Hilbert son los de la forma $4k + 1$ ($1, 5, 9, 13, 17, 21, \dots$): el producto de dos de ellos vuelve a serlo. Un primo de Hilbert es un número de Hilbert mayor que 1 que no es producto de dos números de Hilbert menores. Obtener los primos de Hilbert menores que $N = 10^5$ con una criba que tache los productos de dos números de Hilbert, y comprobar que coinciden con los que predice la teoría: los primos $p \equiv 1 \pmod 4$ y los productos $p \cdot q$ de dos primos (no necesariamente distintos) con $p \equiv q \equiv 3 \pmod 4$.

In [5]:
def primos_de_hilbert(n):
    compuesto = bytearray(n + 1)
    for a in range(5, math.isqrt(n) + 1, 4):
        for b in range(a, n // a + 1, 4):
            compuesto[a * b] = 1
    return [h for h in range(5, n + 1, 4) if not compuesto[h]]


N = 100_000
hilbert = primos_de_hilbert(N)
es = criba(N)
primos_4k3 = [p for p in range(3, N // 3 + 1, 4) if es[p]]
teoricos = {p for p in range(5, N + 1, 4) if es[p]}
teoricos |= {p * q for i, p in enumerate(primos_4k3) for q in primos_4k3[i:] if p * q <= N}
assert set(hilbert) == teoricos
print(len(hilbert), "primos de Hilbert menores que 10⁵")
print("Los 20 primeros:", hilbert[:20])
print("No son primos ordinarios:", [h for h in hilbert[:40] if not es[h]])

10325 primos de Hilbert menores que 10⁵
Los 20 primeros: [5, 9, 13, 17, 21, 29, 33, 37, 41, 49, 53, 57, 61, 69, 73, 77, 89, 93, 97, 101]
No son primos ordinarios: [9, 21, 33, 49, 57, 69, 77, 93, 121, 129, 133, 141, 161, 177, 201, 209, 213, 217]


## 5. La factorización de Hilbert no es única

Escribir `factorizaciones_hilbert(n)`, que devuelva todas las formas de escribir un número de Hilbert como producto de primos de Hilbert (listas en orden no decreciente), recursivamente: probar cada primo de Hilbert $h$ que divide a $n$, no menor que el último usado, y factorizar $n / h$. Encontrar el menor número de Hilbert con dos factorizaciones distintas, listar los cinco primeros y el menor con tres. Comprobar que cada factorización multiplica $n$ y solo usa primos de Hilbert, y que los números de Hilbert que son primos ordinarios tienen una sola.

In [6]:
PRIMOS_H = set(hilbert)


def factorizaciones_hilbert(n, minimo=5):
    if n == 1:
        return [[]]
    resultado = []
    # Basta probar h ≤ √n y el propio n; n / h vuelve a ser 4k + 1
    for h in [*range(minimo, math.isqrt(n) + 1, 4), n]:
        if n % h == 0 and h in PRIMOS_H:
            resultado += [[h, *resto] for resto in factorizaciones_hilbert(n // h, h)]
    return resultado


varias = []
for n in range(5, 20_000, 4):
    fs = factorizaciones_hilbert(n)
    assert fs and all(math.prod(f) == n and all(h in PRIMOS_H for h in f) for f in fs)
    if es[n]:
        assert fs == [[n]]
    if len(fs) > 1:
        varias.append((n, fs))

for n, fs in varias[:5]:
    print(f"{n:>5} = " + " = ".join(" · ".join(map(str, f)) for f in fs))
n, fs = next((n, fs) for n, fs in varias if len(fs) >= 3)
print(f"Primero con {len(fs)}: {n} = " + " = ".join(" · ".join(map(str, f)) for f in fs))
assert varias[0][0] == 441 == 21 * 21 == 9 * 49

  441 = 9 · 49 = 21 · 21
  693 = 9 · 77 = 21 · 33
 1089 = 9 · 121 = 33 · 33
 1197 = 9 · 133 = 21 · 57
 1449 = 9 · 161 = 21 · 69
Primero con 3: 4389 = 21 · 209 = 33 · 133 = 57 · 77


## 6. Sucesión de Thue-Morse

La serie de Thue-Morse empieza en $[0]$ y cada paso concatena lo que hay con su complemento: $[0], [0, 1], [0, 1, 1, 0], [0, 1, 1, 0, 1, 0, 0, 1], \dots$ Calcular así sus primeros $2^{14}$ términos y obtenerlos de otras dos formas: el término $n$ es la paridad del número de unos de $n$ en binario (`int.bit_count`), y cumple $t_0 = 0$, $t_{2n} = t_n$, $t_{2n+1} = 1 - t_n$. Comprobar que las tres coinciden y que el prefijo de longitud $2^k$ tiene exactamente $2^{k-1}$ unos.

In [7]:
def serie_thue_morse(pasos):
    s = [0]
    for _ in range(pasos):
        s += [1 - x for x in s]
    return s


def thue_morse_bits(n):
    return [i.bit_count() % 2 for i in range(n)]


def thue_morse_recurrencia(n):
    t = [0] * n
    for i in range(1, n):
        t[i] = t[i // 2] if i % 2 == 0 else 1 - t[i // 2]
    return t


for k in range(5):
    print(f"paso {k}: {''.join(map(str, serie_thue_morse(k)))}")

tm = serie_thue_morse(14)
assert tm == thue_morse_bits(2**14) == thue_morse_recurrencia(2**14)
assert all(sum(tm[: 2**k]) == 2 ** (k - 1) for k in range(1, 15))
print("Las tres construcciones coinciden en", len(tm), "términos")

paso 0: 0
paso 1: 01
paso 2: 0110
paso 3: 01101001
paso 4: 0110100110010110
Las tres construcciones coinciden en 16384 términos


## 7. Thue-Morse no tiene cubos

Un cubo es un bloque que aparece tres veces seguidas ($www$, como `000` o `101010`). Escribir `primer_cubo(s)`, que busque por fuerza bruta un cubo en una secuencia y devuelva su posición y el bloque, o `None`. Comprobar que el prefijo de longitud 2048 de Thue-Morse no tiene ninguno, y que, en cambio, 1000 secuencias binarias aleatorias de longitud 64 contienen todas al menos un cubo. Thue-Morse sí tiene cuadrados ($ww$): comprobar en su prefijo de 512 términos que las longitudes de $w$ hasta 64 que aparecen son exactamente $2^k$ y $3 \cdot 2^k$.

In [8]:
def primer_cubo(s):
    n = len(s)
    for largo in range(1, n // 3 + 1):
        for i in range(n - 3 * largo + 1):
            w = s[i : i + largo]
            if s[i + largo : i + 2 * largo] == w == s[i + 2 * largo : i + 3 * largo]:
                return i, w
    return None


def largos_de_cuadrados(s, maximo):
    return [largo for largo in range(1, maximo + 1)
            if any(s[i : i + largo] == s[i + largo : i + 2 * largo] for i in range(len(s) - 2 * largo + 1))]


prefijo = "".join(map(str, tm[:2048]))
assert primer_cubo(prefijo) is None
largos = largos_de_cuadrados(prefijo[:512], 64)
print("Thue-Morse (2048 términos): sin cubos")
print("Largos de w en los cuadrados ww (prefijo de 512):", largos)
assert largos == sorted(m * 2**k for m in (1, 3) for k in range(7) if m * 2**k <= 64)

random.seed(17)
aleatorias = ["".join(random.choice("01") for _ in range(64)) for _ in range(1000)]
cubos = [primer_cubo(s) for s in aleatorias]
assert all(cubos)
print("Todas las aleatorias tienen un cubo; bloque del primer cubo hallado (se buscan primero los cortos):", Counter(w for _, w in cubos))

Thue-Morse (2048 términos): sin cubos
Largos de w en los cuadrados ww (prefijo de 512): [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64]
Todas las aleatorias tienen un cubo; bloque del primer cubo hallado (se buscan primero los cortos): Counter({'0': 515, '1': 485})


## 8. Huecos maximales entre primos

Un hueco entre primos consecutivos es maximal si es mayor que todos los anteriores. Con una criba hasta $10^7$, listar los huecos maximales con el primo tras el que aparecen y comprobarlos contra los valores conocidos (OEIS A005250 y A002386). La conjetura de Cramér dice que el hueco que sigue a $p$ es $O((\log p)^2)$: añadir a la tabla el cociente $\text{hueco} / (\ln p)^2$ y comprobar que, a partir de $p = 23$, nunca supera 1 (para los primeros primos el logaritmo es demasiado pequeño).

In [9]:
es = criba(10**7)
record, maximales, anterior = 0, [], 2
for p in range(3, 10**7 + 1, 2):
    if es[p]:
        if p - anterior > record:
            record = p - anterior
            maximales.append((anterior, record))
        anterior = p

print(f"{'hueco':>6} {'tras el primo':>14} {'hueco/(ln p)²':>15}")
for p, g in maximales:
    print(f"{g:>6} {p:>14,} {g / math.log(p) ** 2:>15.3f}")

conocidos_huecos = [1, 2, 4, 6, 8, 14, 18, 20, 22, 34, 36, 44, 52, 72, 86, 96, 112, 114, 118, 132, 148, 154]
conocidos_primos = [2, 3, 7, 23, 89, 113, 523, 887, 1129, 1327, 9551, 15683, 19609, 31397, 155921,
                    360653, 370261, 492113, 1349533, 1357201, 2010733, 4652353]
assert maximales == list(zip(conocidos_primos, conocidos_huecos))
assert all(g < math.log(p) ** 2 for p, g in maximales if p >= 23)

 hueco  tras el primo   hueco/(ln p)²
     1              2           2.081
     2              3           1.657
     4              7           1.056
     6             23           0.610
     8             89           0.397
    14            113           0.626
    18            523           0.459
    20            887           0.434
    22          1,129           0.445
    34          1,327           0.658
    36          9,551           0.429
    44         15,683           0.471
    52         19,609           0.532
    72         31,397           0.672
    86        155,921           0.602
    96        360,653           0.586
   112        370,261           0.681
   114        492,113           0.664
   118      1,349,533           0.592
   132      1,357,201           0.662
   148      2,010,733           0.703
   154      4,652,353           0.653


## 9. Función indicatriz de Euler

$\varphi(n)$ cuenta los enteros de $1$ a $n$ coprimos con $n$. Calcularla de tres formas: por la definición con `math.gcd`; con la fórmula del producto $\varphi(n) = n \prod_{p \mid n} \left(1 - \frac1p\right)$ sobre los primos que dividen a $n$ (con aritmética entera); y para todos los $n \le N$ a la vez con una criba que, para cada primo $p$, multiplique sus múltiplos por $(p-1)/p$. Comprobar que coinciden hasta $N = 5000$, que $\varphi(mn) = \varphi(m)\varphi(n)$ cuando $m$ y $n$ son coprimos, que $\sum_{d \mid n} \varphi(d) = n$ y que $\varphi(n) = n - 1$ solo para los primos.

In [10]:
def phi_definicion(n):
    return sum(math.gcd(k, n) == 1 for k in range(1, n + 1))


def phi_producto(n):
    resultado = n
    for p in factoriza(n):
        resultado = resultado // p * (p - 1)
    return resultado


def phi_criba(n):
    phi = list(range(n + 1))
    for p in range(2, n + 1):
        if phi[p] == p:  # nadie lo ha tocado: es primo
            for m in range(p, n + 1, p):
                phi[m] = phi[m] // p * (p - 1)
    return phi


N = 5000
PHI = phi_criba(N)
assert all(PHI[n] == phi_producto(n) == phi_definicion(n) for n in range(1, N + 1))
assert all(PHI[m * n] == PHI[m] * PHI[n] for m in range(1, 71) for n in range(1, 71) if math.gcd(m, n) == 1)
assert all(sum(PHI[d] for d in range(1, n + 1) if n % d == 0) == n for n in range(1, 1001))
assert all((PHI[n] == n - 1) == es_primo(n) for n in range(2, N + 1))

print("φ(1..20):", PHI[1:21])
for n in (36, 2026, 4096, 600_851_475_143):
    print(f"φ({n}) = {phi_producto(n)}  factores: {dict(factoriza(n))}")

φ(1..20): [1, 1, 2, 2, 4, 2, 6, 4, 6, 4, 10, 4, 12, 6, 8, 8, 16, 6, 18, 8]
φ(36) = 12  factores: {2: 2, 3: 2}
φ(2026) = 1012  factores: {2: 1, 1013: 1}
φ(4096) = 2048  factores: {2: 12}
φ(600851475143) = 591194251200  factores: {71: 1, 839: 1, 1471: 1, 6857: 1}


## 10. Fórmula de Legendre: ceros finales y exponentes

El exponente del primo $p$ en $n!$ es $v_p(n!) = \sum_{i \ge 1} \lfloor n / p^i \rfloor$ (fórmula de Legendre), que también vale $\frac{n - s_p(n)}{p - 1}$, con $s_p(n)$ la suma de las cifras de $n$ en base $p$. Escribir:

- `exponente(x, n)`: el mayor $e$ con $x^e \mid n$, para cualquier $x \ge 2$.
- `legendre(p, n)` y `legendre_cifras(p, n)`: $v_p(n!)$ de las dos formas.
- `ceros_finales(n, base=10)`: cuántos ceros acaba $n!$ en esa base, que es el mínimo de $\lfloor v_p(n!) / a \rfloor$ para cada $p^a$ de la factorización de la base (en base 10, $v_5(n!)$).

Comprobarlo con `math.factorial`: contando los ceros de `str` en base 10, convirtiendo a otras bases y con `exponente` sobre el propio factorial.

In [11]:
def exponente(x, n):
    e = 0
    while n % x == 0:
        n //= x
        e += 1
    return e


def legendre(p, n):
    total, potencia = 0, p
    while potencia <= n:
        total += n // potencia
        potencia *= p
    return total


def suma_cifras(n, base):
    s = 0
    while n:
        n, r = divmod(n, base)
        s += r
    return s


def legendre_cifras(p, n):
    return (n - suma_cifras(n, p)) // (p - 1)


def ceros_finales(n, base=10):
    return min(legendre(p, n) // a for p, a in factoriza(base).items())


def cifras_en_base(n, base):
    cifras = []
    while n:
        n, r = divmod(n, base)
        cifras.append(r)
    return cifras[::-1] or [0]


for n in range(1, 1001):
    f = math.factorial(n)
    s = str(f)
    assert ceros_finales(n) == len(s) - len(s.rstrip("0"))
    for p in (2, 3, 5, 7, 11):
        assert legendre(p, n) == legendre_cifras(p, n) == exponente(p, f)
    if n <= 300:
        for base in (2, 6, 12, 16, 36):
            cifras = cifras_en_base(f, base)
            ceros = len(cifras) - max(i for i, c in enumerate(cifras) if c) - 1
            assert ceros_finales(n, base) == ceros

print(f"{'n':>12} {'ceros de n!':>12} {'base 12':>8} {'base 16':>8}")
for n in (10, 25, 100, 2026, 10**9):
    print(f"{n:>12} {ceros_finales(n):>12} {ceros_finales(n, 12):>8} {ceros_finales(n, 16):>8}")
print(f"2026! acaba en {ceros_finales(2026, 6)} ceros en base 6; el exponente de 12 en 2⁹·3⁴·5 es {exponente(12, 2**9 * 3**4 * 5)}")
assert exponente(12, 2**9 * 3**4 * 5) == 4 and all(exponente(6, math.factorial(n)) == ceros_finales(n, 6) for n in range(1, 200))

           n  ceros de n!  base 12  base 16
          10            2        4        2
          25            6       10        5
         100           24       48       24
        2026          505     1009      504
  1000000000    249999998 499999993 249999996
2026! acaba en 1010 ceros en base 6; el exponente de 12 en 2⁹·3⁴·5 es 4


## 11. Primos cubanos

La diferencia entre dos cubos consecutivos es $(n+1)^3 - n^3 = 3n^2 + 3n + 1$; cuando es un primo se llama primo cubano ($7 = 2^3 - 1^3$, $19 = 3^3 - 2^3$, $37$, $61$…). Escribir un generador infinito `primos_cubanos()` que recorra $n$ con `itertools.count` y obtener los 20 primeros con `islice`, junto con la pareja de cubos de la que salen. Comprobarlo al revés: tomar todos los primos menores que $10^6$ de la criba y quedarse con los que cumplen que $12p - 3$ es un cuadrado perfecto $m^2$ con $m \equiv 3 \pmod 6$ (despejando $n = (m - 3)/6$ de $3n^2 + 3n + 1 = p$).

In [12]:
def primos_cubanos():
    for n in count(1):
        p = (n + 1) ** 3 - n**3
        if es_primo(p):
            yield n, p


for n, p in islice(primos_cubanos(), 6):
    print(f"{n + 1}³ - {n}³ = {p}")
print("Los 20 primeros:", [p for _, p in islice(primos_cubanos(), 20)])


def es_cubano(p):
    m = math.isqrt(12 * p - 3)
    return m * m == 12 * p - 3 and m % 6 == 3


es = criba(10**6)
desde_criba = [p for p in range(2, 10**6) if es[p] and es_cubano(p)]
desde_generador = [p for _, p in islice(primos_cubanos(), len(desde_criba) + 1)]
assert desde_generador[:-1] == desde_criba and desde_generador[-1] >= 10**6
print(len(desde_criba), "primos cubanos menores que 10⁶")

2³ - 1³ = 7
3³ - 2³ = 19
4³ - 3³ = 37
5³ - 4³ = 61
7³ - 6³ = 127
10³ - 9³ = 271
Los 20 primeros: [7, 19, 37, 61, 127, 271, 331, 397, 547, 631, 919, 1657, 1801, 1951, 2269, 2437, 2791, 3169, 3571, 4219]
173 primos cubanos menores que 10⁶


## 12. Cuadrado más cercano

Escribir `cuadrado_mas_cercano(n)`, que devuelva el cuadrado perfecto más próximo a $n \ge 0$ usando `math.isqrt` para obtener $r = \lfloor \sqrt n \rfloor$ y comparando $r^2$ con $(r+1)^2$ (nunca hay empate, porque el punto medio $r^2 + r + \frac12$ no es entero). Comprobarlo con fuerza bruta hasta $10^4$ y con números de 46 cifras, donde la versión con `round(math.sqrt(n)) ** 2` falla porque un `float` solo guarda unas 16 cifras.

In [13]:
def cuadrado_mas_cercano(n):
    r = math.isqrt(n)
    return r * r if n - r * r <= (r + 1) ** 2 - n else (r + 1) ** 2


def cuadrado_mas_cercano_float(n):
    return round(math.sqrt(n)) ** 2


def cuadrado_mas_cercano_bruta(n):
    return min((k * k for k in range(n + 2)), key=lambda c: abs(c - n))


assert all(cuadrado_mas_cercano(n) == cuadrado_mas_cercano_bruta(n) for n in range(10_001))


def es_el_mas_cercano(c, n):
    r = math.isqrt(c)
    return r * r == c and abs(c - n) <= min(abs((r - 1) ** 2 - n), abs((r + 1) ** 2 - n))


random.seed(46)
fallos_float = 0
for _ in range(1000):
    n = random.randrange(10**45, 10**46)
    assert es_el_mas_cercano(cuadrado_mas_cercano(n), n)
    fallos_float += cuadrado_mas_cercano_float(n) != cuadrado_mas_cercano(n)
print(f"Con float fallan {fallos_float} de 1000 números de 46 cifras")
for n in (10**46, 10**46 - 1, 10**46 + 10**23 + 1, 2026):
    c = cuadrado_mas_cercano(n)
    print(f"{n} → {math.isqrt(c)}²  (float: {math.isqrt(cuadrado_mas_cercano_float(n))}²)")

Con float fallan 1000 de 1000 números de 46 cifras
10000000000000000000000000000000000000000000000 → 100000000000000000000000²  (float: 99999999999999991611392²)
9999999999999999999999999999999999999999999999 → 100000000000000000000000²  (float: 99999999999999991611392²)
10000000000000000000000100000000000000000000001 → 100000000000000000000001²  (float: 99999999999999991611392²)
2026 → 45²  (float: 45²)


## 13. Sumar y multiplicar números escritos como cadenas

Escribir `suma_cadenas(a, b)` para dos naturales en base 10 escritos como cadenas, sin convertirlos a `int`: recorrer las cifras de derecha a izquierda con `zip_longest` (rellenando con `"0"`), sumar con acarreo y considerar que la cadena vacía vale 0. Con ella, escribir `producto_cadenas(a, b)` por el método escolar: multiplicar `a` por cada cifra de `b` (con su propio acarreo), añadir tantos ceros como la posición de la cifra y acumular las sumas. Comprobar ambas con `int` en 2000 casos aleatorios de hasta 60 cifras.

In [14]:
def suma_cadenas(a, b):
    cifras, acarreo = [], 0
    for x, y in zip_longest(reversed(a), reversed(b), fillvalue="0"):
        acarreo, cifra = divmod(int(x) + int(y) + acarreo, 10)
        cifras.append(str(cifra))
    if acarreo:
        cifras.append(str(acarreo))
    return "".join(reversed(cifras)).lstrip("0") or "0"


def por_cifra(a, c):
    cifras, acarreo = [], 0
    for x in reversed(a):
        acarreo, cifra = divmod(int(x) * c + acarreo, 10)
        cifras.append(str(cifra))
    return (str(acarreo) if acarreo else "") + "".join(reversed(cifras))


def producto_cadenas(a, b):
    total = "0"
    for posicion, c in enumerate(reversed(b)):
        total = suma_cadenas(total, por_cifra(a, int(c)) + "0" * posicion)
    return total


print(suma_cadenas("2026", "999"), suma_cadenas("", "7"), suma_cadenas("", ""), suma_cadenas("99999", "1"))
print(producto_cadenas("2026", "1496"), producto_cadenas("123456789", "0"))

random.seed(13)
for _ in range(2000):
    a = str(random.randrange(10 ** random.randint(1, 60)))
    b = str(random.randrange(10 ** random.randint(1, 60)))
    assert suma_cadenas(a, b) == str(int(a) + int(b))
    assert producto_cadenas(a, b) == str(int(a) * int(b))

3025 7 0 100000
3030896 0


## 14. Sistema factorádico y la $k$-ésima permutación

En el sistema factorádico la cifra de la posición $i$ (contando desde 0 por la derecha) vale entre $0$ e $i$ y pesa $i!$: $463 = 3 \cdot 5! + 4 \cdot 4! + 1 \cdot 3! + 0 \cdot 2! + 1 \cdot 1! + 0 \cdot 0!$, que se escribe `341010`. Escribir `a_factoradico(n)`, que divida sucesivamente entre $1, 2, 3, \dots$ guardando los restos (cifras mayores que 9 con letras), y `de_factoradico(s)`, la inversa. Comprobar la ida y vuelta hasta $10^5$ y que los números de $0$ a $6! - 1$ dan, sin repetir, todas las secuencias de 6 cifras válidas.

Las cifras factorádicas de $k$ son el código de Lehmer de la $k$-ésima permutación en orden lexicográfico (empezando en 0): cada cifra dice qué elemento de los que quedan hay que tomar. Escribir `permutacion_k(elementos, k)` y su inversa `rango(permutacion)` y comprobarlas con `itertools.permutations`.

In [15]:
DIGITOS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"


def cifras_factoradicas(n):
    cifras, base = [], 1
    while True:
        n, r = divmod(n, base)
        cifras.append(r)
        if n == 0:
            return cifras[::-1]
        base += 1


def a_factoradico(n):
    return "".join(DIGITOS[c] for c in cifras_factoradicas(n))


def de_factoradico(s):
    return sum(DIGITOS.index(c) * math.factorial(i) for i, c in enumerate(reversed(s)))


for n in (0, 5, 463, 2026, 10**30):
    print(f"{n} → {a_factoradico(n)}")
assert a_factoradico(463) == "341010"
assert all(de_factoradico(a_factoradico(n)) == n for n in range(100_000))
assert de_factoradico(a_factoradico(10**30)) == 10**30

n = 6
validas = set(product(*(range(i + 1) for i in reversed(range(n)))))
obtenidas = [tuple(cifras_factoradicas(k)) for k in range(math.factorial(n))]
obtenidas = [(0,) * (n - len(c)) + c for c in obtenidas]
assert len(set(obtenidas)) == len(obtenidas) == len(validas) and set(obtenidas) == validas


def permutacion_k(elementos, k):
    quedan = sorted(elementos)
    cifras = cifras_factoradicas(k)
    cifras = [0] * (len(quedan) - len(cifras)) + cifras
    return [quedan.pop(c) for c in cifras]


def rango(permutacion):
    quedan, k = sorted(permutacion), 0
    for i, x in enumerate(permutacion):
        k += quedan.index(x) * math.factorial(len(permutacion) - 1 - i)
        quedan.remove(x)
    return k


elementos = list("CANRIO")
for k, p in enumerate(permutations(sorted(elementos))):
    assert permutacion_k(elementos, k) == list(p) and rango(p) == k
print("Permutación 463 de CANRIO:", "".join(permutacion_k(elementos, 463)))
print("Permutación 999 999 de 0..9:", "".join(map(str, permutacion_k(range(10), 999_999))))

0 → 0
5 → 210
463 → 341010
2026 → 2441200
1000000000000000000000000000000 → 37MFCDG58464CF8BC011822512200
Permutación 463 de CANRIO: NRCAOI
Permutación 999 999 de 0..9: 2783915460


## 15. Triángulo de los impares

Los impares se colocan en filas de longitud creciente: $1$; $3\ 5$; $7\ 9\ 11$; $13\ 15\ 17\ 19$… Escribir `fila(n)`, que calcule directamente la fila $n$ sabiendo que antes hay $\frac{n(n-1)}{2}$ impares, y comprobar que coincide con la construcción por trozos de un generador de impares. Comprobar que la suma de la fila $n$ es $n^3$ y, como consecuencia, que la suma de las $n$ primeras filas, $1^3 + \dots + n^3$, es el cuadrado del número triangular $\left(\frac{n(n+1)}{2}\right)^2$.

In [16]:
def fila(n):
    primero = n * (n - 1) + 1
    return list(range(primero, primero + 2 * n, 2))


impares = count(1, 2)
filas = [list(islice(impares, n)) for n in range(1, 201)]
assert all(fila(n) == filas[n - 1] for n in range(1, 201))
for n in range(1, 6):
    print(f"{' '.join(map(str, fila(n))):>16}   suma = {sum(fila(n)):>3} = {n}³")

assert all(sum(fila(n)) == n**3 for n in range(1, 2001))
sumas_filas = accumulate(sum(fila(n)) for n in range(1, 2001))
assert all(s == (n * (n + 1) // 2) ** 2 for n, s in enumerate(sumas_filas, 1))

               1   suma =   1 = 1³
             3 5   suma =   8 = 2³
          7 9 11   suma =  27 = 3³
     13 15 17 19   suma =  64 = 4³
  21 23 25 27 29   suma = 125 = 5³


## 16. Sumar consecutivos hasta que quede uno

A partir de una lista, formar otra con las sumas de cada pareja de elementos consecutivos y repetir hasta que quede un solo número: $[1, 2, 3, 4] \to [3, 5, 7] \to [8, 12] \to [20]$. Escribir `reitera_sumas(xs)` con `itertools.pairwise`, guardando las filas, y obtener el mismo resultado sin iterar: el número final es $\sum_i \binom{n-1}{i} x_i$, porque cada $x_i$ llega por tantos caminos como en el triángulo de Pascal. Comprobarlo con 500 listas aleatorias.

In [17]:
def reitera_sumas(xs):
    filas = [list(xs)]
    while len(filas[-1]) > 1:
        filas.append([a + b for a, b in pairwise(filas[-1])])
    return filas


def suma_binomial(xs):
    n = len(xs)
    return sum(math.comb(n - 1, i) * x for i, x in enumerate(xs))


xs = [1, 2, 3, 4, 5, 6]
for f in reitera_sumas(xs):
    print(" ".join(f"{x:>3}" for x in f).center(26))
print("Coeficientes:", [math.comb(len(xs) - 1, i) for i in range(len(xs))], "→", suma_binomial(xs))

random.seed(16)
for _ in range(500):
    xs = [random.randint(-50, 50) for _ in range(random.randint(1, 25))]
    assert reitera_sumas(xs)[-1] == [suma_binomial(xs)]

   1   2   3   4   5   6  
     3   5   7   9  11    
       8  12  16  20      
        20  28  36        
          48  64          
           112            
Coeficientes: [1, 5, 10, 10, 5, 1] → 112


## 17. Reconocer potencias de 4

Escribir `es_potencia_de_4(n)` sin bucles: $n$ es potencia de 4 si es positivo, tiene un único bit a 1 (`n & (n - 1) == 0`) y ese bit está en una posición par (`(n.bit_length() - 1) % 2 == 0`). Para enteros de 32 bits vale también la máscara `0x55555555`, que tiene a 1 justo las posiciones pares. Comprobar las versiones con otra que divide entre 4 mientras puede, para todos los $n < 2^{16}$, para $4^k$ y sus vecinos con $k$ hasta 300, y para $2 \cdot 4^k$, que son potencias de 2 pero no de 4.

In [18]:
def es_potencia_de_4(n):
    return n > 0 and n & (n - 1) == 0 and (n.bit_length() - 1) % 2 == 0


def es_potencia_de_4_mascara(n):
    return 0 < n < 2**32 and n & (n - 1) == 0 and n & 0x55555555 != 0


def es_potencia_de_4_bucle(n):
    if n <= 0:
        return False
    while n % 4 == 0:
        n //= 4
    return n == 1


for n in (1, 2, 8, 16, 96, 1024, 2048):
    print(f"{n:>5} {n:>12b} {es_potencia_de_4(n)}")

assert all(es_potencia_de_4(n) == es_potencia_de_4_mascara(n) == es_potencia_de_4_bucle(n) for n in range(-5, 2**16))
for k in range(301):
    for n in (4**k - 1, 4**k, 4**k + 1, 2 * 4**k):
        assert es_potencia_de_4(n) == es_potencia_de_4_bucle(n)
        if n < 2**32:
            assert es_potencia_de_4_mascara(n) == es_potencia_de_4_bucle(n)
print("4³⁰⁰ es potencia de 4:", es_potencia_de_4(4**300), "| 2·4³⁰⁰:", es_potencia_de_4(2 * 4**300))

    1            1 True
    2           10 False
    8         1000 False
   16        10000 True
   96      1100000 False
 1024  10000000000 True
 2048 100000000000 False
4³⁰⁰ es potencia de 4: True | 2·4³⁰⁰: False


## 18. Máximos locales y duplicar elementos

Escribir dos funciones de listas, cada una de dos formas, y comprobar que dan lo mismo:

- `maximos_locales(xs)`: los elementos estrictamente mayores que su anterior y su siguiente (ni el primero ni el último lo son), con un bucle de índices y con una comprensión sobre `zip(xs, xs[1:], xs[2:])`. Aplicarla a las temperaturas máximas de una quincena en La Laguna.
- `duplica(xs)`: repite cada elemento dos veces, con una comprensión doble y con `itertools.chain.from_iterable` y `repeat`; si recibe una cadena, devuelve una cadena (`"Teide"` → `"TTeeiiddee"`).

In [19]:
def maximos_locales(xs):
    return [xs[i] for i in range(1, len(xs) - 1) if xs[i - 1] < xs[i] > xs[i + 1]]


def maximos_locales_zip(xs):
    return [b for a, b, c in zip(xs, xs[1:], xs[2:]) if a < b > c]


def duplica(xs):
    dobles = [x for x in xs for _ in range(2)]
    return "".join(dobles) if isinstance(xs, str) else dobles


def duplica_itertools(xs):
    dobles = chain.from_iterable(repeat(x, 2) for x in xs)
    return "".join(dobles) if isinstance(xs, str) else list(dobles)


temperaturas = [21, 23, 22, 22, 24, 26, 25, 25, 27, 27, 26, 24, 25, 23, 23]
print("Máximos locales:", maximos_locales(temperaturas))
print(duplica([1, 2, 3]), duplica("Teide"), repr(duplica("")))

random.seed(18)
for _ in range(1000):
    xs = [random.randint(0, 5) for _ in range(random.randint(0, 12))]
    assert maximos_locales(xs) == maximos_locales_zip(xs)
    assert duplica(xs) == duplica_itertools(xs) and len(duplica(xs)) == 2 * len(xs)
    s = "".join(map(str, xs))
    assert duplica(s) == duplica_itertools(s) and duplica(s)[::2] == s

Máximos locales: [23, 26, 25]
[1, 1, 2, 2, 3, 3] TTeeiiddee ''
